<a href="https://colab.research.google.com/github/Chaki34/SIH-2026-Indian-Railways-needs-to-stop-block-a-railway-track-for-maintenance-PS-26027/blob/main/PS-26027.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

# Configuration
NUM_RECORDS = 10000
START_DATE = datetime(2022, 1, 1)
END_DATE = datetime(2026, 12, 31)
OUTPUT_DIR = "railway_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Master Infrastructure Setup
zones = [f"Z{i:02d}" for i in range(1, 3)]
divisions = [f"D{i:02d}" for i in range(1, 5)]
corridors = [f"C{i:02d}" for i in range(1, 10)]
sections = [f"SEC{i:03d}" for i in range(1, 50)]

section_map = []
for sec in sections:
    section_map.append({
        "section_id": sec,
        "corridor_id": random.choice(corridors),
        "division_id": random.choice(divisions),
        "zone_id": random.choice(zones)
    })
df_infra = pd.DataFrame(section_map)

# 2. Asset History Generation
asset_types = ['Track', 'Signal', 'Traction']
assets = []
for i in range(1, 1001):
    a_type = random.choice(asset_types)
    a_id = f"{a_type[:3].upper()}{i:04d}"
    sec = random.choice(sections)
    install_date = START_DATE - timedelta(days=random.randint(365, 3650))
    assets.append({
        "asset_id": a_id,
        "asset_type": a_type,
        "section_id": sec,
        "installation_date": install_date.strftime('%Y-%m-%d'),
        "asset_age_years": (datetime.now() - install_date).days // 365,
        "criticality_score": random.randint(1, 10),
        "condition_score": random.randint(30, 95),
        "last_maintenance_date": (datetime.now() - timedelta(days=random.randint(10, 100))).strftime('%Y-%m-%d'),
        "failure_count_historical": random.randint(0, 15),
        "operational_status": "Active"
    })
df_assets = pd.DataFrame(assets)

# 3. Generate Maintenance History (Logic-Based)
maint_records = []
for i in range(NUM_RECORDS):
    asset = df_assets.sample(1).iloc[0]
    date_rep = START_DATE + timedelta(days=random.randint(0, (END_DATE - START_DATE).days))

    # Logic: Older assets or lower condition scores lead to higher severity
    severity = min(10, max(1, int((100 - asset['condition_score'])/10 + random.randint(-2, 2))))
    crit = asset['criticality_score']
    urgency = min(10, max(1, int((severity + crit)/2 + random.randint(-1, 1))))

    est_dur = round(random.uniform(1.5, 6.0), 1)
    # Target variable for ML: Actual duration
    act_dur = round(est_dur * random.uniform(0.9, 1.4), 1) if random.random() > 0.1 else round(est_dur * 1.8, 1) # 10% blocks are delayed

    maint_records.append({
        "task_id": f"T{i:06d}",
        "asset_id": asset['asset_id'],
        "section_id": asset['section_id'],
        "date_reported": date_rep.strftime('%Y-%m-%d'),
        "maintenance_type": random.choice(["Preventive", "Corrective", "Inspection", "Repair"]),
        "severity": severity,
        "criticality_score": crit,
        "urgency_score": urgency,
        "safety_risk_score": min(10, severity + 1),
        "asset_age_years": asset['asset_age_years'],
        "overdue_days": random.randint(0, 30) if severity < 7 else random.randint(0, 3),
        "estimated_duration_hours": est_dur,
        "actual_duration_hours": act_dur,
        "status": "Completed",
        "completion_date": (date_rep + timedelta(days=2)).strftime('%Y-%m-%d')
    })
df_maint = pd.DataFrame(maint_records)

# 4. Train Operation History (Peak/Off-Peak Traffic)
train_ops = []
for i in range(NUM_RECORDS * 2):
    sec = random.choice(sections)
    date = START_DATE + timedelta(days=random.randint(0, (END_DATE - START_DATE).days))
    hour = random.randint(0, 23)

    # Traffic Density Logic
    if 6 <= hour <= 10 or 17 <= hour <= 21:
        traffic = "High"
        delay_prob = 0.3
    elif 23 <= hour or hour <= 4:
        traffic = "Low"
        delay_prob = 0.05
    else:
        traffic = "Medium"
        delay_prob = 0.15

    sched_arr = f"{hour:02d}:{random.randint(0,59):02d}"
    train_ops.append({
        "record_id": f"TRN{i:07d}",
        "date": date.strftime('%Y-%m-%d'),
        "train_id": f"12{random.randint(100,999)}",
        "train_type": random.choice(["Express", "Superfast", "Goods", "Passenger"]),
        "section_id": sec,
        "scheduled_arrival": sched_arr,
        "actual_runtime_minutes": random.randint(45, 120),
        "traffic_density": traffic,
        "is_goods_train": random.choice([True, False]),
        "passenger_load_factor": round(random.uniform(0.2, 1.0), 2)
    })
df_trains = pd.DataFrame(train_ops)

# 5. Maintenance Blocks (The core planning dataset)
blocks = []
for i in range(2000):
    sec = random.choice(sections)
    date = START_DATE + timedelta(days=random.randint(0, (END_DATE - START_DATE).days))
    start_h = random.choice([23, 0, 1, 11, 12]) # Mostly night, some midday
    duration = random.randint(2, 4)

    # Outcome logic
    tasks_count = random.randint(1, 5)
    trains_affected = random.randint(0, 8) if start_h < 20 and start_h > 5 else random.randint(0, 2)

    blocks.append({
        "block_id": f"BLK{i:05d}",
        "date": date.strftime('%Y-%m-%d'),
        "section_id": sec,
        "block_start": f"{start_h:02d}:00",
        "planned_duration_hours": float(duration),
        "actual_duration_hours": duration + (random.uniform(0, 1.5) if random.random() > 0.8 else 0),
        "departments_involved": random.choice(["Track", "Signal", "Multi"]),
        "trains_affected": trains_affected,
        "total_delay_minutes": trains_affected * random.randint(20, 60),
        "combined_block": random.choice([True, False]),
        "successful_block": random.choice([True, True, True, False]) # 75% success rate
    })
df_blocks = pd.DataFrame(blocks)

# 6. Unified Training Dataset (Merging for ML)
# We join maintenance tasks with their section info and asset info
df_training = df_maint.merge(df_infra, on="section_id", how="left")
df_training = df_training.merge(df_assets[['asset_id', 'condition_score']], on="asset_id", how="left")

# Add some derived target features for training
df_training['priority_label'] = pd.cut(df_training['urgency_score'], bins=[0, 4, 7, 10], labels=['Low', 'Medium', 'High'])

# Exporting Files
df_maint.to_csv(f"{OUTPUT_DIR}/track_maintenance_history.csv", index=False)
df_assets.to_csv(f"{OUTPUT_DIR}/asset_history.csv", index=False)
df_trains.to_csv(f"{OUTPUT_DIR}/train_operation_history.csv", index=False)
df_blocks.to_csv(f"{OUTPUT_DIR}/maintenance_block_history.csv", index=False)
df_infra.to_csv(f"{OUTPUT_DIR}/corridor_availability_history.csv", index=False)
df_training.to_csv(f"{OUTPUT_DIR}/historical_training_dataset.csv", index=False)

# Quality Report
report = {
    "dataset": ["Track", "Assets", "Trains", "Blocks"],
    "row_count": [len(df_maint), len(df_assets), len(df_trains), len(df_blocks)],
    "null_values": [0, 0, 0, 0]
}
pd.DataFrame(report).to_csv(f"{OUTPUT_DIR}/data_quality_report.csv", index=False)

print(f"Success! 10,000+ records generated across multiple files in the '{OUTPUT_DIR}' folder.")

Success! 10,000+ records generated across multiple files in the 'railway_data' folder.
